# Verbalization Baseline Comparison

Computes accuracy for:
- **Baseline** (Zero-shot)
- **LIT**
- **Patchscopes**


In [1]:
import json
import os
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np

# Task names
TASKS = [
    "country_currency",
    "food_from_country",
    "person_plays_position_in_sport",
    "person_plays_pro_sport",
    "product_by_company",
    "star_constellation"
]

# Base paths
BASE_DIR = Path("/home/li.mil/verb_faithfulness/results") # Change this with your path
BASELINE_DIR = BASE_DIR / "baseline" / "zeroshot_llama3"
LIT_DIR = BASE_DIR / "verbalization" / "lit_llama3"
PATCHSCOPES_DIR = BASE_DIR / "verbalization" / "patchscopes_llama3"

## Helper Functions

In [2]:
def load_json_results(filepath):
    """Load JSON results file."""
    if not filepath.exists():
        return None
    with open(filepath, 'r') as f:
        return json.load(f)

def check_correct(completion, answer):
    """Check if completion contains the correct answer (case-insensitive)."""
    if completion is None or answer is None:
        return False
    return answer.lower() in completion.lower()

def compute_accuracy(results):
    """Compute accuracy for a set of results."""
    if results is None or len(results) == 0:
        return None
    
    correct = 0
    total = len(results)
    
    for item in results.values():
        if check_correct(item['completion'], item['answer']):
            correct += 1
    
    return correct / total if total > 0 else 0.0

def get_available_layers(base_dir):
    """Get list of available layer directories."""
    if not base_dir.exists():
        return []
    return sorted([int(d.name) for d in base_dir.iterdir() if d.is_dir() and d.name.isdigit()])

def get_available_patchscopes_configs():
    """Get available patchscopes src/tgt configurations."""
    if not PATCHSCOPES_DIR.exists():
        return []
    configs = []
    for d in PATCHSCOPES_DIR.iterdir():
        if d.is_dir() and 'src=' in d.name and 'tgt=' in d.name:
            parts = d.name.split('_')
            src = int(parts[0].split('=')[1])
            tgt = int(parts[1].split('=')[1])
            configs.append((src, tgt, d.name))
    return sorted(configs)

## Load All Results

In [ ]:
# Load baseline
baseline_results = {}
for task in TASKS:
    filepath = BASELINE_DIR / f"{task}.json"
    baseline_results[task] = load_json_results(filepath)

# Load LIT results for all layers
lit_layers = get_available_layers(LIT_DIR)
lit_results = defaultdict(dict)
for layer in lit_layers:
    for task in TASKS:
        filepath = LIT_DIR / str(layer) / f"{task}.json"
        lit_results[layer][task] = load_json_results(filepath)

# Load Patchscopes results
configs = get_available_patchscopes_configs()
patchscopes_results = defaultdict(lambda: defaultdict(dict))
for src, tgt, dirname in configs:
    for task in TASKS:
        filepath = PATCHSCOPES_DIR / dirname / f"{task}.json"
        patchscopes_results[src][tgt][task] = load_json_results(filepath)

print(f"Loaded:")
print(f"  Baseline: {len([r for r in baseline_results.values() if r is not None])} tasks")
print(f"  LIT: {len(lit_layers)} layers")
print(f"  Patchscopes: {len(configs)} configurations")

Loaded:
  Baseline: 6 tasks
  LIT: 15 layers
  Patchscopes: 465 configurations


## Compute Baseline Accuracy

In [4]:
baseline_acc = {}
for task in TASKS:
    baseline_acc[task] = compute_accuracy(baseline_results[task])

print("\nBASELINE ACCURACY:")
print("="*60)
for task in TASKS:
    if baseline_acc[task] is not None:
        print(f"{task:40s}: {baseline_acc[task]:.4f}")
    else:
        print(f"{task:40s}: No results")

valid_accs = [a for a in baseline_acc.values() if a is not None]
if valid_accs:
    print(f"\n{'AVERAGE':40s}: {np.mean(valid_accs):.4f}")


BASELINE ACCURACY:
country_currency                        : 0.8203
food_from_country                       : 0.5909
person_plays_position_in_sport          : 0.5851
person_plays_pro_sport                  : 0.7575
product_by_company                      : 0.6712
star_constellation                      : 0.4402

AVERAGE                                 : 0.6442


## Compute LIT Accuracy (Averaged Across Layers)

In [10]:
# For each task, compute accuracy for each layer, then average across layers
lit_acc_per_task = {}

for task in TASKS:
    layer_accs = []
    for layer in lit_layers:
        acc = compute_accuracy(lit_results[layer][task])
        if acc is not None:
            layer_accs.append(acc)
    
    lit_acc_per_task[task] = np.mean(layer_accs) if layer_accs else None

print("\nLIT ACCURACY (AVERAGED ACROSS LAYERS):")
print("="*60)
for task in TASKS:
    if lit_acc_per_task[task] is not None:
        print(f"{task:40s}: {lit_acc_per_task[task]:.4f}")
    else:
        print(f"{task:40s}: No results")

valid_accs = [a for a in lit_acc_per_task.values() if a is not None]
if valid_accs:
    print(f"\n{'AVERAGE':40s}: {np.mean(valid_accs):.4f}")


LIT ACCURACY (AVERAGED ACROSS LAYERS):
country_currency                        : 0.7849
food_from_country                       : 0.4455
person_plays_position_in_sport          : 0.6578
person_plays_pro_sport                  : 0.8124
product_by_company                      : 0.5791
star_constellation                      : 0.3308

AVERAGE                                 : 0.6017


## Compute Patchscopes Accuracy

Patchscopes' evaluation is done by patching a single src layers into all tgt layers; if any answer is correct at any tgt layer, then the answer is counted as correct. Then, the accuracy is averaged across src layers.

In [8]:
# Get unique source layers
src_layers = sorted(set([src for src, tgt, _ in configs]))

# For each task and source layer, compute "any correct" accuracy
patchscopes_acc_per_task = {}

for task in TASKS:
    src_accs = []
    
    for src in src_layers:
        # Get all target layers for this source layer
        tgt_layers_for_src = [tgt for s, tgt, _ in configs if s == src]
        
        if not tgt_layers_for_src:
            continue
        
        # Get number of samples (from first available target layer)
        first_tgt = tgt_layers_for_src[0]
        sample_results = patchscopes_results[src][first_tgt][task]
        
        if sample_results is None:
            continue
        
        n_samples = len(sample_results)
        correct_count = 0
        
        # For each sample, check if ANY target layer gets it correct
        for sample_idx in range(n_samples):
            sample_idx_str = str(sample_idx)
            any_correct = False
            
            for tgt in tgt_layers_for_src:
                results = patchscopes_results[src][tgt][task]
                if results is None or sample_idx_str not in results:
                    continue
                
                item = results[sample_idx_str]
                if check_correct(item['completion'], item['answer']):
                    any_correct = True
                    break
            
            if any_correct:
                correct_count += 1
        
        # Accuracy for this source layer
        src_acc = correct_count / n_samples if n_samples > 0 else 0.0
        src_accs.append(src_acc)
    
    # Average across source layers
    patchscopes_acc_per_task[task] = np.mean(src_accs) if src_accs else None
print("\nPATCHSCOPES ACCURACY:")
print("="*60)
for task in TASKS:
    if patchscopes_acc_per_task[task] is not None:
        print(f"{task:40s}: {patchscopes_acc_per_task[task]:.4f}")
    else:
        print(f"{task:40s}: No results")

valid_accs = [a for a in patchscopes_acc_per_task.values() if a is not None]
if valid_accs:
    print(f"\n{'AVERAGE':40s}: {np.mean(valid_accs):.4f}")


PATCHSCOPES ACCURACY:
country_currency                        : 0.4057
food_from_country                       : 0.2742
person_plays_position_in_sport          : 0.5472
person_plays_pro_sport                  : 0.8641
product_by_company                      : 0.4493
star_constellation                      : 0.3797

AVERAGE                                 : 0.4867


## Summary Table

In [7]:
print("\n" + "="*80)
print("SUMMARY: ACCURACY BY TASK AND METHOD")
print("="*80 + "\n")

summary_data = {
    'Baseline': baseline_acc,
    'LIT': lit_acc_per_task,
    'Patchscopes': patchscopes_acc_per_task
}

df = pd.DataFrame(summary_data).T
df['AVERAGE'] = df.mean(axis=1)

print(df.round(4))

print("\n" + "="*80)
print("OVERALL AVERAGES:")
print("="*80)
for method in ['Baseline', 'LIT', 'Patchscopes']:
    avg = df.loc[method, 'AVERAGE']
    print(f"{method:20s}: {avg:.4f}")


SUMMARY: ACCURACY BY TASK AND METHOD

             country_currency  food_from_country  \
Baseline               0.8203             0.5909   
LIT                    0.7849             0.4455   
Patchscopes            0.4057             0.2742   

             person_plays_position_in_sport  person_plays_pro_sport  \
Baseline                             0.5851                  0.7575   
LIT                                  0.6578                  0.8124   
Patchscopes                          0.5472                  0.8641   

             product_by_company  star_constellation  AVERAGE  
Baseline                 0.6712              0.4402   0.6442  
LIT                      0.5791              0.3308   0.6017  
Patchscopes              0.4493              0.3797   0.4867  

OVERALL AVERAGES:
Baseline            : 0.6442
LIT                 : 0.6017
Patchscopes         : 0.4867
